# 03 · Metriche di valutazione

## Obiettivo

Capire e verificare il calcolo delle metriche utilizzate per valutare
la segmentazione degli edifici e la classificazione del loro danno,
partendo dal codice pubblico di BRIGHT.

Una metrica confronta una previsione con la maschera di riferimento:
la previsione indica le classi assegnate dal modello, mentre la maschera
contiene le etichette corrette disponibili nel dataset.

## Attività previste

1. Esaminare il codice delle metriche nel repository BRIGHT.
2. Comprendere la matrice di confusione, IoU, precisione, recall e F1.
3. Verificare i calcoli su piccoli esempi con risultato noto.
4. Valutare previsioni banali sulle maschere locali, per capire
   come si comportano le metriche.
5. Distinguere le metriche per edificio/sfondo da quelle
   per le classi di danno.

## Ambito e limiti

Questo notebook verifica il funzionamento della valutazione.
Non addestra un modello e non riproduce ancora i risultati dei vincitori.

Le prove sui dati locali riguardano soltanto i tre esempi scaricati.
La formula del punteggio della competizione e le classi incluse
nel calcolo dovranno essere verificate prima di confrontare i risultati.

## 1. Lettura del codice di riferimento

Leggiamo il file `bda_benchmark/util_func/metrics.py` del repository
BRIGHT per individuare le metriche implementate e comprenderne il calcolo.

In questo passaggio mostriamo il contenuto del file senza eseguirlo
né modificarlo.

La presenza di una metrica nel file non implica che venga utilizzata
nel punteggio finale della competizione: verificheremo anche come
le funzioni vengono richiamate dagli script di valutazione.

In [1]:
from pathlib import Path

PROGETTO = Path.cwd()

PERCORSO_METRICHE = (
    PROGETTO
    / "BRIGHT"
    / "bda_benchmark"
    / "util_func"
    / "metrics.py"
)

if not PERCORSO_METRICHE.is_file():
    raise FileNotFoundError(
        f"File delle metriche non trovato: {PERCORSO_METRICHE}\n"
        "Verifica che il notebook sia nella cartella riproduzione_dfc2025."
    )

codice_metriche = PERCORSO_METRICHE.read_text(encoding="utf-8")

print("File analizzato:", PERCORSO_METRICHE)
print()
print(codice_metriche)

File analizzato: /Users/antonyogrossi/tesi/riproduzione_dfc2025/BRIGHT/bda_benchmark/util_func/metrics.py

import numpy as np


class Evaluator(object):
    def __init__(self, num_class):
        self.num_class = num_class
        self.confusion_matrix = np.zeros((self.num_class,) * 2, dtype=np.longlong)
        self._epsilon = 1e-7

    def Pixel_Accuracy(self):
        Acc = np.diag(self.confusion_matrix).sum() / self.confusion_matrix.sum()
        return Acc

    def Pixel_Accuracy_Class(self):
        Acc = np.diag(self.confusion_matrix) / (self.confusion_matrix.sum(axis=1) + self._epsilon)
        mAcc = np.nanmean(Acc)
        return mAcc, Acc

    def Pixel_Precision_Rate(self):
        assert self.confusion_matrix.shape[0] == 2
        Pre = self.confusion_matrix[1, 1] / (self.confusion_matrix[0, 1] + self.confusion_matrix[1, 1] + self._epsilon)
        return Pre

    def Pixel_Recall_Rate(self):
        assert self.confusion_matrix.shape[0] == 2
        Rec = self.confusion_m

## 2. La matrice di confusione

La matrice di confusione conta i confronti tra classi di riferimento
e classi previste, pixel per pixel.

Nel codice BRIGHT:
- le righe rappresentano le classi vere;
- le colonne rappresentano le classi previste;
- la diagonale contiene le classificazioni corrette;
- le altre celle contengono gli errori.

Iniziamo con due classi:
0 = sfondo;
1 = edificio.

Usiamo un piccolo esempio artificiale per poter verificare
manualmente tutti i conteggi.

In [2]:
import numpy as np
import pandas as pd

# Etichette corrette di sei pixel
riferimento = np.array([
    [0, 0, 1],
    [1, 1, 0],
])

# Previsioni inventate per gli stessi sei pixel
previsione = np.array([
    [0, 1, 1],
    [0, 1, 0],
])

matrice = np.zeros((2, 2), dtype=np.int64)

for classe_vera, classe_prevista in zip(
    riferimento.ravel(),
    previsione.ravel(),
):
    matrice[classe_vera, classe_prevista] += 1

tabella_confusione = pd.DataFrame(
    matrice,
    index=["Vero sfondo", "Vero edificio"],
    columns=["Previsto sfondo", "Previsto edificio"],
)

display(tabella_confusione)

,Previsto sfondo,Previsto edificio
Vero sfondo,2,1
Vero edificio,1,2


## 3. Metriche per la classe edificio

Consideriamo edificio come classe positiva.

Dalla matrice di confusione ricaviamo:
- TP: pixel edificio riconosciuti correttamente;
- FP: pixel sfondo scambiati per edificio;
- FN: pixel edificio scambiati per sfondo;
- TN: pixel sfondo riconosciuti correttamente.

Calcoliamo:
- precisione = TP / (TP + FP);
- recall = TP / (TP + FN);
- IoU edificio = TP / (TP + FP + FN);
- F1/Dice edificio = 2TP / (2TP + FP + FN);
- accuratezza = (TP + TN) / totale pixel.

L'accuratezza considera entrambe le classi; le altre metriche
qui riportate si riferiscono alla classe edificio.

In questo esempio tutti i denominatori sono maggiori di zero.
La gestione dei casi con denominatore nullo verrà verificata
separatamente nel confronto con il codice BRIGHT.

In [3]:
TN = int(matrice[0, 0])
FP = int(matrice[0, 1])
FN = int(matrice[1, 0])
TP = int(matrice[1, 1])

risultati = {
    "Accuratezza complessiva": (TP + TN) / matrice.sum(),
    "Precisione edificio": TP / (TP + FP),
    "Recall edificio": TP / (TP + FN),
    "IoU edificio": TP / (TP + FP + FN),
    "F1/Dice edificio": 2 * TP / (2 * TP + FP + FN),
}

tabella_metriche = pd.DataFrame({
    "valore": risultati,
})
tabella_metriche["percentuale"] = 100 * tabella_metriche["valore"]

display(tabella_metriche.round(4))

,valore,percentuale
Accuratezza complessiva,0.6667,66.6667
Precisione edificio,0.6667,66.6667
Recall edificio,0.6667,66.6667
IoU edificio,0.5000,50.0000
F1/Dice edificio,0.6667,66.6667


## 4. Confronto con il codice BRIGHT

Carichiamo la classe Evaluator dal file metrics.py del repository.

Utilizziamo le stesse matrici artificiali già analizzate:
riferimento contiene le classi corrette e previsione quelle previste.

Verifichiamo:
- che la matrice di confusione coincida con quella costruita manualmente;
- che accuratezza, precisione, recall, IoU edificio e F1 edificio
  coincidano entro una piccola tolleranza numerica.

Il codice BRIGHT aggiunge piccole costanti ad alcuni denominatori.
Sono quindi possibili differenze minime rispetto alle formule
calcolate direttamente.

In [4]:
import importlib.util

# Carichiamo il file Python tramite il suo percorso.
specifica = importlib.util.spec_from_file_location(
    "bright_metrics",
    PERCORSO_METRICHE,
)
modulo_metriche = importlib.util.module_from_spec(specifica)
specifica.loader.exec_module(modulo_metriche)

# Due classi: 0 = sfondo, 1 = edificio.
valutatore = modulo_metriche.Evaluator(num_class=2)

# Confrontiamo riferimento e previsione.
valutatore.add_batch(riferimento, previsione)

print("Matrice di confusione calcolata da BRIGHT:")
print(valutatore.confusion_matrix)

assert np.array_equal(
    matrice,
    valutatore.confusion_matrix,
), "Le matrici di confusione non coincidono."

risultati_bright = {
    "Accuratezza complessiva": valutatore.Pixel_Accuracy(),
    "Precisione edificio": valutatore.Pixel_Precision_Rate(),
    "Recall edificio": valutatore.Pixel_Recall_Rate(),
    "IoU edificio": valutatore.Intersection_over_Union()[1],
    "F1/Dice edificio": valutatore.Pixel_F1_score(),
}

confronto = pd.DataFrame({
    "nostro_calcolo": risultati,
    "BRIGHT": risultati_bright,
})

confronto["differenza_assoluta"] = (
    confronto["nostro_calcolo"] - confronto["BRIGHT"]
).abs()

display(confronto)

assert np.allclose(
    confronto["nostro_calcolo"],
    confronto["BRIGHT"],
    rtol=0,
    atol=1e-6,
), "Le metriche non coincidono entro la tolleranza."

print("Controllo superato: matrice e metriche coincidono entro la tolleranza.")

Matrice di confusione calcolata da BRIGHT:
[[2 1]
 [1 2]]


,nostro_calcolo,BRIGHT,differenza_assoluta
Accuratezza complessiva,0.666667,0.666667,0.000000e+00
Precisione edificio,0.666667,0.666667,2.222222e-08
Recall edificio,0.666667,0.666667,2.222222e-08
IoU edificio,0.500000,0.500000,1.250000e-08
F1/Dice edificio,0.666667,0.666667,2.222222e-08


Controllo superato: matrice e metriche coincidono entro la tolleranza.


## 5. Caso limite: previsione sempre sfondo

Costruiamo una previsione che assegna la classe 0 a tutti i pixel.

Poiché il riferimento contiene edifici, questa previsione non riconosce
nessun pixel edificio:
- TP = 0;
- FP = 0;
- FN > 0.

La recall e la IoU edificio sono zero.
La precisione ha denominatore nullo, perché non è stato previsto
alcun pixel edificio: il suo valore dipende dalla convenzione adottata.

Controlliamo come il codice BRIGHT gestisce questo caso,
in particolare nel calcolo dell'F1 binario.

In [5]:
# Stessa forma del riferimento, ma tutti i valori sono 0.
previsione_sfondo = np.zeros_like(riferimento)

valutatore_sfondo = modulo_metriche.Evaluator(num_class=2)
valutatore_sfondo.add_batch(riferimento, previsione_sfondo)

display(pd.DataFrame(
    valutatore_sfondo.confusion_matrix,
    index=["Vero sfondo", "Vero edificio"],
    columns=["Previsto sfondo", "Previsto edificio"],
))

# In questo test prevediamo una divisione 0/0 nell'F1 binario.
# Nascondiamo il relativo avviso, ma conserviamo il risultato NaN.
with np.errstate(divide="ignore", invalid="ignore"):
    risultati_sfondo = {
        "Accuratezza complessiva": valutatore_sfondo.Pixel_Accuracy(),
        "Precisione edificio": valutatore_sfondo.Pixel_Precision_Rate(),
        "Recall edificio": valutatore_sfondo.Pixel_Recall_Rate(),
        "IoU edificio": valutatore_sfondo.Intersection_over_Union()[1],
        "F1 binario BRIGHT": valutatore_sfondo.Pixel_F1_score(),
    }

display(pd.DataFrame({
    "valore": risultati_sfondo,
}))

,Previsto sfondo,Previsto edificio
Vero sfondo,3,0
Vero edificio,3,0


,valore
Accuratezza complessiva,0.5
Precisione edificio,0.0
Recall edificio,0.0
IoU edificio,0.0
F1 binario BRIGHT,NaN


## 6. F1 calcolato direttamente dai conteggi

Calcoliamo l'F1 della classe edificio con la formula:

F1 = 2TP / (2TP + FP + FN)

Questa forma restituisce zero quando non viene riconosciuto alcun
pixel edificio, ma sono presenti edifici nel riferimento.

Se invece la classe edificio è assente sia nel riferimento sia
nella previsione, il denominatore è zero. In questa funzione
restituiamo NaN per segnalare che il punteggio non è definito.

Questa è una scelta esplicita della nostra funzione diagnostica.
Non modifichiamo il codice BRIGHT; per la valutazione finale
dovremo rispettare la convenzione del protocollo adottato.

In [6]:
def f1_edificio_da_matrice(matrice_confusione):
    TP = int(matrice_confusione[1, 1])
    FP = int(matrice_confusione[0, 1])
    FN = int(matrice_confusione[1, 0])

    denominatore = 2 * TP + FP + FN

    if denominatore == 0:
        return np.nan

    return 2 * TP / denominatore


confronto_f1 = pd.DataFrame({
    "caso": [
        "Previsione iniziale",
        "Previsione sempre sfondo",
    ],
    "F1 BRIGHT": [
        risultati_bright["F1/Dice edificio"],
        risultati_sfondo["F1 binario BRIGHT"],
    ],
    "F1 dai conteggi": [
        f1_edificio_da_matrice(valutatore.confusion_matrix),
        f1_edificio_da_matrice(valutatore_sfondo.confusion_matrix),
    ],
}).set_index("caso")

display(confronto_f1)

,F1 BRIGHT,F1 dai conteggi
caso,,
Previsione iniziale,0.666667,0.666667
Previsione sempre sfondo,NaN,0.000000


## 7. Matrice di confusione per le quattro classi

Passiamo alla classificazione del danno:

- 0: sfondo;
- 1: edificio integro;
- 2: edificio danneggiato;
- 3: edificio distrutto.

La matrice di confusione ha quattro righe e quattro colonne.
Le righe indicano le classi vere, le colonne quelle previste.

Utilizziamo otto pixel artificiali, con due pixel di riferimento
per ciascuna classe, per poter verificare i conteggi manualmente.

In [7]:
nomi_classi = [
    "Sfondo",
    "Integro",
    "Danneggiato",
    "Distrutto",
]

riferimento_danno = np.array([
    [0, 0, 1, 1],
    [2, 2, 3, 3],
])

previsione_danno = np.array([
    [0, 1, 1, 1],
    [2, 0, 3, 2],
])

valutatore_danno = modulo_metriche.Evaluator(num_class=4)
valutatore_danno.add_batch(riferimento_danno, previsione_danno)

tabella_danno = pd.DataFrame(
    valutatore_danno.confusion_matrix,
    index=[f"Vero {nome}" for nome in nomi_classi],
    columns=[f"Previsto {nome}" for nome in nomi_classi],
)

display(tabella_danno)

# Conteggi attesi, ottenuti confrontando a mano gli otto pixel.
matrice_attesa = np.array([
    [1, 1, 0, 0],
    [0, 2, 0, 0],
    [1, 0, 1, 0],
    [0, 0, 1, 1],
])

assert np.array_equal(
    valutatore_danno.confusion_matrix,
    matrice_attesa,
), "La matrice non coincide con i conteggi attesi."

print("Controllo superato: i conteggi delle quattro classi sono corretti.")

,Previsto Sfondo,Previsto Integro,Previsto Danneggiato,Previsto Distrutto
Vero Sfondo,1,1,0,0
Vero Integro,0,2,0,0
Vero Danneggiato,1,0,1,0
Vero Distrutto,0,0,1,1


Controllo superato: i conteggi delle quattro classi sono corretti.


## 8. IoU e F1 per le classi del danno

Per ogni classe ricaviamo dalla matrice di confusione:

- TP: elemento sulla diagonale;
- FN: somma della riga meno TP;
- FP: somma della colonna meno TP.

Calcoliamo quindi:
- IoU = TP / (TP + FP + FN);
- F1 = 2TP / (2TP + FP + FN).

La funzione BRIGHT Intersection_over_Union restituisce una IoU
per tutte le classi, compreso lo sfondo.

Damage_F1_score restituisce invece gli F1 delle classi 1, 2 e 3.
Gli errori che coinvolgono lo sfondo restano inclusi nei conteggi:
viene escluso soltanto il punteggio della classe 0 dall'output.

Verifichiamo i risultati confrontandoli con il calcolo diretto.
In questo esempio tutti i denominatori sono maggiori di zero.

In [8]:
matrice_danno = valutatore_danno.confusion_matrix

# Un conteggio per ciascuna delle quattro classi.
TP = np.diag(matrice_danno)
FN = matrice_danno.sum(axis=1) - TP
FP = matrice_danno.sum(axis=0) - TP

iou_diretta = TP / (TP + FP + FN)
f1_diretto = 2 * TP / (2 * TP + FP + FN)

iou_bright = valutatore_danno.Intersection_over_Union()
f1_danno_bright = valutatore_danno.Damage_F1_score()

tabella_metriche_danno = pd.DataFrame({
    "TP": TP,
    "FP": FP,
    "FN": FN,
    "IoU diretta": iou_diretta,
    "IoU BRIGHT": iou_bright,
    "F1 diretto": f1_diretto,
    "F1 danno BRIGHT": [np.nan, *f1_danno_bright],
}, index=nomi_classi)

display(tabella_metriche_danno.round(4))

assert np.allclose(
    iou_diretta,
    iou_bright,
    rtol=0,
    atol=1e-6,
)

assert np.allclose(
    f1_diretto[1:],
    f1_danno_bright,
    rtol=0,
    atol=1e-6,
)

print("Controllo superato: IoU e F1 coincidono entro la tolleranza.")

,TP,FP,FN,IoU diretta,IoU BRIGHT,F1 diretto,F1 danno BRIGHT
Sfondo,1,1,1,0.3333,0.3333,0.5000,NaN
Integro,2,1,0,0.6667,0.6667,0.8000,0.8000
Danneggiato,1,1,1,0.3333,0.3333,0.5000,0.5000
Distrutto,1,0,1,0.5000,0.5000,0.6667,0.6667


Controllo superato: IoU e F1 coincidono entro la tolleranza.


## 9. Media delle IoU: quali classi includiamo?

La mean IoU, abbreviata mIoU, è la media delle IoU delle classi
considerate. Ogni classe contribuisce alla media con lo stesso peso.

Confrontiamo:
- la media sulle quattro classi, incluso lo sfondo;
- la media sulle tre classi di edificio, quindi escludendo lo sfondo.

La funzione Mean_Intersection_over_Union di BRIGHT calcola la media
su tutte le classi configurate nel valutatore.

Escludere lo sfondo dalla media non significa ignorare gli errori
che lo coinvolgono: un edificio previsto come sfondo resta un falso
negativo per la sua classe.

Questo confronto non identifica ancora il punteggio ufficiale
della competizione: occorre verificare quali classi e quale procedura
di aggregazione vengono effettivamente utilizzate.

In [9]:
# Le quattro IoU, nell'ordine: sfondo, integro, danneggiato, distrutto.
iou_per_classe = valutatore_danno.Intersection_over_Union()

media_quattro_classi = np.mean(iou_per_classe)
media_classi_edificio = np.mean(iou_per_classe[1:])

media_funzione_bright = (
    valutatore_danno.Mean_Intersection_over_Union()
)

tabella_medie = pd.DataFrame({
    "valore": {
        "Media sulle quattro classi": media_quattro_classi,
        "Media sulle tre classi edificio": media_classi_edificio,
        "Mean_Intersection_over_Union BRIGHT": media_funzione_bright,
    }
})

tabella_medie["percentuale"] = 100 * tabella_medie["valore"]

display(tabella_medie.round(4))

assert np.isclose(
    media_quattro_classi,
    media_funzione_bright,
    rtol=0,
    atol=1e-6,
)

print("Verificato: in questo caso la media BRIGHT include lo sfondo.")

,valore,percentuale
Media sulle quattro classi,0.4583,45.8333
Media sulle tre classi edificio,0.5000,50.0000
Mean_Intersection_over_Union BRIGHT,0.4583,45.8333


Verificato: in questo caso la media BRIGHT include lo sfondo.


## 10. Ricerca delle metriche negli script BRIGHT

Cerchiamo dove vengono richiamate le funzioni di valutazione
nel codice del benchmark.

L'obiettivo è individuare:
- quali metriche vengono calcolate;
- se vengono selezionate solo alcune classi;
- come vengono aggregati i risultati.

La ricerca mostra file e righe pertinenti, senza eseguire gli script.
Le corrispondenze saranno poi lette nel loro contesto:
da sole non dimostrano quale sia il protocollo ufficiale del contest.

In [10]:
CARTELLA_BENCHMARK = PROGETTO / "BRIGHT" / "bda_benchmark"

termini = [
    "Evaluator(",
    "Intersection_over_Union(",
    "Damage_F1_score(",
    "Pixel_F1_score(",
    "add_batch(",
    ".reset(",
]

for percorso in sorted(CARTELLA_BENCHMARK.rglob("*.py")):
    # Abbiamo già letto il file che definisce le metriche.
    if percorso == PERCORSO_METRICHE:
        continue

    righe = percorso.read_text(encoding="utf-8").splitlines()

    corrispondenze = [
        (numero, riga.strip())
        for numero, riga in enumerate(righe, start=1)
        if any(termine in riga for termine in termini)
    ]

    if corrispondenze:
        print(f"\n--- {percorso.relative_to(PROGETTO)} ---")

        for numero, testo in corrispondenze:
            print(f"{numero}: {testo}")


--- BRIGHT/bda_benchmark/script/cross_event/one_shot/train_DamageFormer.py ---
42: self.evaluator = Evaluator(num_class=4)
173: self.evaluator.reset()
196: self.evaluator.add_batch(labels_clf, output_clf)
200: IoU_of_each_class = self.evaluator.Intersection_over_Union()
201: mIoU = self.evaluator.Mean_Intersection_over_Union()
208: self.evaluator.reset()
231: self.evaluator.add_batch(labels_clf, output_clf)
235: IoU_of_each_class = self.evaluator.Intersection_over_Union()
236: mIoU = self.evaluator.Mean_Intersection_over_Union()
243: self.evaluator.reset()
266: self.evaluator.add_batch(labels_clf, output_clf)
269: IoU_of_each_class = self.evaluator.Intersection_over_Union()
270: mIoU = self.evaluator.Mean_Intersection_over_Union()

--- BRIGHT/bda_benchmark/script/cross_event/one_shot/train_MeanTeacher.py ---
45: self.evaluator = Evaluator(num_class=4)
247: self.evaluator.reset()
270: self.evaluator.add_batch(labels_clf, output_clf)
274: IoU_of_each_class = self.evaluator.Intersection_

### Lettura del contesto: valutazione nello script U-Net

Lo script train_UNet.py utilizza valutatori distinti per localizzazione,
classificazione del danno e valutazione complessiva.

Leggiamo il blocco di valutazione per verificare:
- come vengono preparate le etichette e le previsioni;
- quali pixel vengono passati ai diversi valutatori;
- quando vengono azzerati e accumulati i conteggi;
- come vengono combinati i punteggi.

Questo controllo riguarda lo script del benchmark BRIGHT.
Non identifica automaticamente il protocollo della competizione.

In [11]:
PERCORSO_SCRIPT = (
    CARTELLA_BENCHMARK
    / "script"
    / "standard_ML"
    / "train_UNet.py"
)

righe_script = PERCORSO_SCRIPT.read_text(
    encoding="utf-8"
).splitlines()

inizio = 140
fine = 195

print("File:", PERCORSO_SCRIPT.relative_to(PROGETTO))
print()

for numero in range(inizio, min(fine, len(righe_script)) + 1):
    print(f"{numero:3}: {righe_script[numero - 1]}")

File: BRIGHT/bda_benchmark/script/standard_ML/train_UNet.py

140:                             'clf f1 (test)': harmonic_mean_f1_test * 100,
141:                             'OA (test)': final_OA_test * 100,
142:                             'mIoU (test)': mIoU_test * 100,
143:                             'sub class IoU (test)': IoU_of_each_class_test * 100
144:                         }
145:                     self.deep_model.train()
146: 
147:         print('The accuracy of the best round is ', best_round)
148: 
149: 
150:     def validation(self):
151:         print('---------starting validation-----------')
152:         self.evaluator_total.reset()
153:         self.evaluator_loc.reset()
154:         self.evaluator_clf.reset()
155:         val_dataset = MultimodalDamageAssessmentDatset(self.args.val_dataset_path, self.args.val_data_name_list, 1024, None, 'test')
156:         val_data_loader = DataLoader(val_dataset, batch_size=self.args.eval_batch_size, num_workers=1, drop_last=Fals

### Risultato dell'ispezione del codice

Nel metodo validation() di train_UNet.py sono presenti tre valutazioni:

1. Localizzazione: le classi previste 1, 2 e 3 vengono riunite
   nella classe edificio e confrontate con il riferimento binario.
   Viene calcolato l'F1 edificio.

2. Danno: vengono selezionati soltanto i pixel che il riferimento
   labels_loc indica come edificio. Su questi pixel vengono calcolati
   gli F1 delle tre classi di edificio, combinati mediante media armonica.
   Un edificio previsto come sfondo resta un errore; i pixel di vero
   sfondo sono invece esclusi da questa specifica valutazione.

3. Mappa completa: riferimento e previsione vengono confrontati
   senza la selezione precedente. Si calcolano accuratezza, IoU
   per classe e media delle quattro IoU, sfondo incluso.

I conteggi vengono accumulati su tutti i batch della validation.
Le metriche sono calcolate al termine, anziché mediate per immagine.

Queste conclusioni descrivono il metodo validation() dello script
ispezionato. Non dimostrano, da sole, quale punteggio sia stato
utilizzato per la classifica ufficiale DFC2025.

## 11. Valutazione del danno sui pixel edificio del riferimento

Riproduciamo sul piccolo esempio artificiale la selezione osservata
nel metodo validation() di train_UNet.py.

Manteniamo soltanto le posizioni in cui il riferimento indica
una delle classi edificio: 1, 2 o 3.

Confrontiamo gli F1 delle classi di edificio:
- calcolati sulla mappa completa;
- calcolati soltanto sui pixel edificio del riferimento.

La selezione viene utilizzata per il calcolo della metrica:
non modifica la previsione prodotta dal modello.

In [12]:
# True nelle posizioni che il riferimento indica come edificio.
selezione_edifici = riferimento_danno > 0

riferimento_selezionato = riferimento_danno[selezione_edifici]
previsione_selezionata = previsione_danno[selezione_edifici]

print("Riferimento selezionato:", riferimento_selezionato)
print("Previsione selezionata: ", previsione_selezionata)

valutatore_danno_selezionato = modulo_metriche.Evaluator(num_class=4)
valutatore_danno_selezionato.add_batch(
    riferimento_selezionato,
    previsione_selezionata,
)

display(pd.DataFrame(
    valutatore_danno_selezionato.confusion_matrix,
    index=[f"Vero {nome}" for nome in nomi_classi],
    columns=[f"Previsto {nome}" for nome in nomi_classi],
))

f1_mappa_completa = valutatore_danno.Damage_F1_score()
f1_pixel_edificio = valutatore_danno_selezionato.Damage_F1_score()

confronto_selezione = pd.DataFrame({
    "F1 sulla mappa completa": f1_mappa_completa,
    "F1 sui veri pixel edificio": f1_pixel_edificio,
}, index=nomi_classi[1:])

display(confronto_selezione.round(4))

Riferimento selezionato: [1 1 2 2 3 3]
Previsione selezionata:  [1 1 2 0 3 2]


,Previsto Sfondo,Previsto Integro,Previsto Danneggiato,Previsto Distrutto
Vero Sfondo,0,0,0,0
Vero Integro,0,2,0,0
Vero Danneggiato,1,0,1,0
Vero Distrutto,0,0,1,1


,F1 sulla mappa completa,F1 sui veri pixel edificio
Integro,0.8000,1.0000
Danneggiato,0.5000,0.5000
Distrutto,0.6667,0.6667


## 12. Media armonica degli F1 del danno

Il metodo validation() di train_UNet.py combina gli F1 delle tre
classi edificio mediante media armonica:

H = 3 / (1/F1_integro + 1/F1_danneggiato + 1/F1_distrutto)

Riproduciamo questa operazione sui punteggi ottenuti selezionando
i pixel edificio del riferimento.

La confrontiamo con la media aritmetica per mostrare che le due
aggregazioni producono risultati diversi.

In questo esempio tutti gli F1 sono positivi, quindi i reciproci
sono definiti. Il valore ottenuto descrive il nostro esempio
artificiale, non le prestazioni di un modello su BRIGHT.

In [13]:
# F1 delle tre classi, calcolati nella cella precedente.
f1_classi = f1_pixel_edificio

# Verifichiamo le condizioni di questo specifico esempio.
assert np.all(np.isfinite(f1_classi))
assert np.all(f1_classi > 0)

# Stessa formula utilizzata nel blocco BRIGHT ispezionato.
media_armonica = len(f1_classi) / np.sum(1.0 / f1_classi)

media_aritmetica = np.mean(f1_classi)

tabella_aggregazione = pd.DataFrame({
    "valore": {
        "Media aritmetica F1": media_aritmetica,
        "Media armonica F1": media_armonica,
    }
})

tabella_aggregazione["percentuale"] = (
    100 * tabella_aggregazione["valore"]
)

display(tabella_aggregazione.round(4))

assert np.isclose(
    media_armonica,
    2 / 3,
    rtol=0,
    atol=1e-6,
)

print("Controllo superato: media armonica coerente con il calcolo manuale.")

,valore,percentuale
Media aritmetica F1,0.7222,72.2222
Media armonica F1,0.6667,66.6667


Controllo superato: media armonica coerente con il calcolo manuale.


## 13. Lettura delle maschere BRIGHT locali

Carichiamo le maschere del danno presenti nella cartella
dati/bright_esempi/target.

Per ogni maschera controlliamo:
- la presenza di una sola banda;
- le dimensioni;
- il numero di pixel validi;
- le classi presenti.

I pixel dichiarati non validi nel raster vengono esclusi.
Lo sfondo, classe 0, viene mantenuto se non è dichiarato NoData.

Le maschere saranno utilizzate come riferimento per valutare
previsioni artificiali di controllo. Non stiamo ancora eseguendo
un modello sulle immagini.

In [14]:
import rasterio

CARTELLA_MASCHERE = (
    PROGETTO / "dati" / "bright_esempi" / "target"
)

percorsi_maschere = sorted(
    CARTELLA_MASCHERE.glob("*_building_damage.tif")
)

if not percorsi_maschere:
    raise FileNotFoundError(
        f"Nessuna maschera trovata in: {CARTELLA_MASCHERE}"
    )

maschere_locali = {}
righe_controllo = []

for percorso in percorsi_maschere:
    identificativo = percorso.name.removesuffix(
        "_building_damage.tif"
    )

    with rasterio.open(percorso) as raster:
        if raster.count != 1:
            raise ValueError(
                f"{identificativo}: attesa una sola banda."
            )

        maschera = raster.read(1, masked=True)

    pixel_validi = maschera.compressed()

    if pixel_validi.size == 0:
        raise ValueError(
            f"{identificativo}: nessun pixel valido."
        )

    classi_presenti = np.unique(pixel_validi)

    if not np.all(np.isin(classi_presenti, [0, 1, 2, 3])):
        raise ValueError(
            f"{identificativo}: classi inattese {classi_presenti}"
        )

    maschere_locali[identificativo] = maschera

    righe_controllo.append({
        "id": identificativo,
        "righe": maschera.shape[0],
        "colonne": maschera.shape[1],
        "pixel_validi": pixel_validi.size,
        "classi_presenti": classi_presenti.tolist(),
    })

controllo_maschere = pd.DataFrame(
    righe_controllo
).set_index("id")

display(controllo_maschere)

print("Maschere caricate:", len(maschere_locali))

,righe,colonne,pixel_validi,classi_presenti
id,,,,
bata-explosion_00000014,1024,1024,1048576,"[0, 1]"
bata-explosion_00000047,1024,1024,1048576,"[0, 1]"
bata-explosion_00000049,1024,1024,1048576,"[0, 1, 2, 3]"


Maschere caricate: 3


## 14. Previsioni di controllo sulle maschere reali

Valutiamo due previsioni artificiali:
- perfetta: identica alla maschera di riferimento;
- sempre sfondo: classe 0 per tutti i pixel.

Per ciascun caso accumuliamo una matrice di confusione sulle tre
maschere, utilizzando soltanto i pixel validi.

Calcoliamo l'accuratezza, le quattro IoU e la loro media mediante
il codice BRIGHT. Nel campione complessivo sono presenti tutte
le classi.

Queste prove verificano il funzionamento della valutazione
sui dati reali: non rappresentano risultati di una rete.

In [15]:
valutatori_controllo = {
    "Perfetta": modulo_metriche.Evaluator(num_class=4),
    "Sempre sfondo": modulo_metriche.Evaluator(num_class=4),
}

for identificativo, maschera in maschere_locali.items():
    # Sequenza delle etichette dei soli pixel validi.
    riferimento_valido = maschera.compressed().astype(np.int64)

    previsioni_controllo = {
        "Perfetta": riferimento_valido.copy(),
        "Sempre sfondo": np.zeros_like(riferimento_valido),
    }

    for caso, previsione_controllo in previsioni_controllo.items():
        valutatori_controllo[caso].add_batch(
            riferimento_valido,
            previsione_controllo,
        )

# Calcoliamo le metriche dopo aver accumulato tutte le immagini.
righe_risultati = []

for caso, valutatore_controllo in valutatori_controllo.items():
    iou = valutatore_controllo.Intersection_over_Union()

    righe_risultati.append({
        "caso": caso,
        "accuratezza": valutatore_controllo.Pixel_Accuracy(),
        "IoU sfondo": iou[0],
        "IoU integro": iou[1],
        "IoU danneggiato": iou[2],
        "IoU distrutto": iou[3],
        "mIoU quattro classi": (
            valutatore_controllo.Mean_Intersection_over_Union()
        ),
    })

risultati_controllo = pd.DataFrame(
    righe_risultati
).set_index("caso")

print("Metriche espresse in percentuale:")
display((100 * risultati_controllo).round(2))

# Controlli sul caso perfetto.
assert np.allclose(
    valutatori_controllo["Perfetta"].Intersection_over_Union(),
    np.ones(4),
    rtol=0,
    atol=1e-6,
)

# La previsione sempre sfondo non riconosce nessuna classe edificio.
assert np.allclose(
    valutatori_controllo["Sempre sfondo"].Intersection_over_Union()[1:],
    np.zeros(3),
    rtol=0,
    atol=1e-6,
)

print("Controlli superati.")

Metriche espresse in percentuale:


,accuratezza,IoU sfondo,IoU integro,IoU danneggiato,IoU distrutto,mIoU quattro classi
caso,,,,,,
Perfetta,100.00,100.00,100.0,100.0,100.0,100.00
Sempre sfondo,88.91,88.91,0.0,0.0,0.0,22.23


Controlli superati.


## 15. Classi assenti e media per immagine

Calcoliamo la mIoU di una previsione perfetta separatamente
per ciascuna maschera.

Nell'implementazione BRIGHT, una classe assente sia nel riferimento
sia nella previsione riceve IoU zero. La media sulle quattro classi
include anche questi zeri.

Confrontiamo poi:
- la media aritmetica delle mIoU delle singole immagini;
- la mIoU calcolata dalla matrice di confusione accumulata.

Sono due procedure diverse e possono produrre risultati diversi.
Il metodo validation() ispezionato utilizza la seconda.

In [16]:
righe_per_immagine = []

for identificativo, maschera in maschere_locali.items():
    riferimento_valido = maschera.compressed().astype(np.int64)

    valutatore_singolo = modulo_metriche.Evaluator(num_class=4)
    valutatore_singolo.add_batch(
        riferimento_valido,
        riferimento_valido.copy(),
    )

    iou = valutatore_singolo.Intersection_over_Union()

    righe_per_immagine.append({
        "id": identificativo,
        "IoU sfondo": iou[0],
        "IoU integro": iou[1],
        "IoU danneggiato": iou[2],
        "IoU distrutto": iou[3],
        "mIoU": valutatore_singolo.Mean_Intersection_over_Union(),
    })

metriche_perfette_per_immagine = pd.DataFrame(
    righe_per_immagine
).set_index("id")

print("Previsioni perfette: metriche in percentuale")
display((100 * metriche_perfette_per_immagine).round(2))

media_miou_immagini = metriche_perfette_per_immagine["mIoU"].mean()

miou_accumulata = (
    valutatori_controllo["Perfetta"].Mean_Intersection_over_Union()
)

print(f"Media delle mIoU per immagine: {100 * media_miou_immagini:.2f}%")
print(f"mIoU dalla matrice accumulata: {100 * miou_accumulata:.2f}%")

Previsioni perfette: metriche in percentuale


,IoU sfondo,IoU integro,IoU danneggiato,IoU distrutto,mIoU
id,,,,,
bata-explosion_00000014,100.0,100.0,0.0,0.0,50.0
bata-explosion_00000047,100.0,100.0,0.0,0.0,50.0
bata-explosion_00000049,100.0,100.0,100.0,100.0,100.0


Media delle mIoU per immagine: 66.67%
mIoU dalla matrice accumulata: 100.00%


## 16. Esportazione dei risultati

Salviamo in formato CSV le principali tabelle prodotte:

- confronto tra calcoli diretti e codice BRIGHT;
- comportamento dell'F1 nel caso sempre sfondo;
- metriche per le quattro classi;
- confronto tra medie e tra selezioni dei pixel;
- controllo delle maschere locali;
- risultati delle previsioni artificiali;
- effetto delle classi assenti sulla valutazione per immagine.

Salviamo anche le matrici di confusione accumulate sulle maschere
reali, per rendere consultabili i conteggi da cui derivano le metriche.

Le colonne dei punteggi sono esportate nella scala 0–1;
le colonne esplicitamente chiamate percentuale sono nella scala 0–100.

I valori non definiti o non applicabili sono indicati con NaN.

I file vengono scritti in risultati/03_metriche.
Rieseguire la cella aggiorna i CSV con i risultati presenti in memoria.

In [17]:
CARTELLA_RISULTATI_03 = PROGETTO / "risultati" / "03_metriche"
CARTELLA_RISULTATI_03.mkdir(parents=True, exist_ok=True)

tabelle_da_salvare = {
    "confronto_metriche_binarie.csv": confronto,
    "confronto_f1_caso_limite.csv": confronto_f1,
    "metriche_quattro_classi.csv": tabella_metriche_danno,
    "confronto_medie_iou.csv": tabella_medie,
    "confronto_selezione_pixel.csv": confronto_selezione,
    "confronto_aggregazione_f1.csv": tabella_aggregazione,
    "controllo_maschere_locali.csv": controllo_maschere,
    "previsioni_controllo.csv": risultati_controllo,
    "previsione_perfetta_per_immagine.csv": (
        metriche_perfette_per_immagine
    ),
}

# Conserviamo anche il confronto tra le due aggregazioni.
tabelle_da_salvare["confronto_aggregazione_miou.csv"] = pd.DataFrame({
    "valore": {
        "Media delle mIoU per immagine": media_miou_immagini,
        "mIoU dalla matrice accumulata": miou_accumulata,
    }
})

# Aggiungiamo i conteggi accumulati dei due esperimenti.
for caso, valutatore_controllo in valutatori_controllo.items():
    nome_file = (
        "matrice_confusione_"
        + caso.lower().replace(" ", "_")
        + ".csv"
    )

    tabelle_da_salvare[nome_file] = pd.DataFrame(
        valutatore_controllo.confusion_matrix,
        index=[f"Vero {nome}" for nome in nomi_classi],
        columns=[f"Previsto {nome}" for nome in nomi_classi],
    )

for nome_file, tabella in tabelle_da_salvare.items():
    tabella.to_csv(
        CARTELLA_RISULTATI_03 / nome_file,
        index=True,
        na_rep="NaN",
    )

print("Cartella dei risultati:", CARTELLA_RISULTATI_03)
print("Tabelle esportate:", len(tabelle_da_salvare))

for nome_file in sorted(tabelle_da_salvare):
    print("-", nome_file)

Cartella dei risultati: /Users/antonyogrossi/tesi/riproduzione_dfc2025/risultati/03_metriche
Tabelle esportate: 12
- confronto_aggregazione_f1.csv
- confronto_aggregazione_miou.csv
- confronto_f1_caso_limite.csv
- confronto_medie_iou.csv
- confronto_metriche_binarie.csv
- confronto_selezione_pixel.csv
- controllo_maschere_locali.csv
- matrice_confusione_perfetta.csv
- matrice_confusione_sempre_sfondo.csv
- metriche_quattro_classi.csv
- previsione_perfetta_per_immagine.csv
- previsioni_controllo.csv


## 17. Conclusioni

### Obiettivo raggiunto

Abbiamo verificato il funzionamento delle principali metriche
di segmentazione, partendo dal codice pubblico di BRIGHT.

Il lavoro ha seguito tre passaggi:
1. calcoli su piccoli esempi artificiali verificabili manualmente;
2. confronto con la classe Evaluator del repository;
3. applicazione a previsioni di controllo sulle maschere locali.

### Risultati delle verifiche

La matrice di confusione utilizza le righe per le classi vere
e le colonne per quelle previste.

Su esempi con denominatori non nulli, i nostri calcoli di accuratezza,
precisione, recall, IoU e F1 coincidono con quelli di BRIGHT entro
la tolleranza numerica adottata.

Sono emersi due comportamenti importanti dell'implementazione:

- Pixel_F1_score restituisce NaN quando precisione e recall sono
  entrambe zero. Nel caso con edifici reali ma previsione sempre
  sfondo, la formula diretta basata su TP, FP e FN restituisce zero.
- Una classe assente sia nel riferimento sia nella previsione
  riceve IoU zero e contribuisce comunque alla media calcolata
  da Mean_Intersection_over_Union.

Il codice originale BRIGHT non è stato modificato.

### Procedura osservata nello script U-Net

Nel metodo validation() di
bda_benchmark/script/standard_ML/train_UNet.py abbiamo verificato:

- valutazione binaria edificio/sfondo mediante F1;
- valutazione del danno sui soli pixel indicati come edificio
  dal riferimento, con media armonica degli F1 delle tre classi;
- valutazione della mappa completa mediante accuratezza,
  IoU per classe e media delle quattro IoU, sfondo incluso.

I conteggi vengono accumulati su tutti i batch prima di calcolare
le metriche finali.

La selezione dei pixel edificio del riferimento esclude gli errori
sul vero sfondo da quello specifico calcolo del danno.
Questi errori restano considerati nelle valutazioni binaria
e complessiva.

### Prove sulle maschere locali

Abbiamo valutato una previsione perfetta e una previsione sempre
sfondo sulle maschere disponibili, accumulando i conteggi.

La previsione perfetta raggiunge punteggi prossimi a 1 per tutte
le classi presenti nel campione complessivo.

La previsione sempre sfondo raggiunge un'accuratezza pari alla
frequenza dello sfondo, ma ha IoU zero per tutte le classi edificio.
Questo mostra perché l'accuratezza complessiva non basta a giudicare
la qualità della segmentazione.

Abbiamo inoltre verificato che la media delle mIoU delle singole
immagini può differire dalla mIoU della matrice accumulata.
Nelle prove effettuate, la differenza emerge anche con previsioni
perfette, a causa delle classi assenti in alcune immagini.

### Risultati salvati

Le tabelle dei confronti e le matrici di confusione delle prove
sulle maschere locali sono esportate in:

risultati/03_metriche

I valori aggiornati sono consultabili negli output del notebook
e nei file CSV.

### Limiti e verifiche ancora necessarie

Le previsioni utilizzate sono artificiali: nessuna rete è stata
addestrata o eseguita per produrle.

Le prove locali riguardano un piccolo campione dello stesso evento
e non misurano le prestazioni di un modello sull'intero dataset.

L'ispezione dello script del benchmark non dimostra quale procedura
sia stata utilizzata per la classifica ufficiale DFC2025.
Prima di confrontare risultati sperimentali con quelli pubblicati,
sarà necessario confermare:

- le suddivisioni e le etichette utilizzate;
- le classi e i pixel inclusi nella valutazione;
- il trattamento di classi assenti e valori non definiti;
- il metodo di aggregazione e il punteggio finale.

### Passaggio successivo

Prima esecuzione di un modello su un campione limitato,
verificando l'intera procedura:
caricamento dei dati, produzione delle previsioni e valutazione.

### Riferimenti del codice

- BRIGHT/bda_benchmark/util_func/metrics.py
- BRIGHT/bda_benchmark/script/standard_ML/train_UNet.py

Repository: https://github.com/ChenHongruixuan/BRIGHT